# Second account: M0 seeds 1-2 + M2 SegFormer (one seed)
Runtime: **L4 GPU**. Click *Runtime > Run all*, approve Drive access, then leave the tab open. About 4.5 h, roughly 20-22 units.

Order and why:
1. Setup + data copy (15 min).
2. **M2 smoke test** (5 min): catches errors early. Check its output after 20 min: it should print `epoch 0:`.
3. **M0 seeds 1 and 2** (about 50 min each): make the M0 vs M1 comparison rest on 3 seeds, like the loss comparison.
4. **M2 full run** (about 1-1.5 h, optional): siamese SegFormer, weighted CE, seed 0, preliminary.
5. Disconnect (stops billing).

A failed training command does not stop the queue. Results go to `checkpoints/` on the shared Drive.
If this account has fewer than about 22 units, delete the M2 full-run cell before Run all.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv   # expect NVIDIA L4

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [4]:
from google.colab import drive
drive.mount("/content/drive")
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
!ls "{CK}"     # if this fails: the shared folder is not shared with / shortcut-added to this account

Mounted at /content/drive
m0_early_fusion_unet_s0  m1_siamese_unet_ce_s2	m1_siamese_unet_s2
m1_siamese_unet_ce_s0	 m1_siamese_unet_s0	smoke_m1_s0
m1_siamese_unet_ce_s1	 m1_siamese_unet_s1


In [5]:
%cd /content
!rm -rf /content/FloodVerify
!git clone -q -b m3/eval-export https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!git log --oneline -1
!pip install -q -e . --no-deps jsonschema

/content
/content/FloodVerify
a1f9090 (HEAD -> m3/eval-export, origin/m3/eval-export) Calibration (temperature, ECE), evaluate CLI, predicted evidence export with crop stitching
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for floodsitrep (pyproject.toml) ... done


In [6]:
# Data to local disk (10-15 min)
!mkdir -p /content/xbd
!tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
!tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
!cp -r "{D}/xbd/hold" /content/xbd/
!cp -r "{D}/xbd/tier3" /content/xbd/
!cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
!ls /content/xbd

hold  rscc_flood_manifest.csv  test  tier3  train


## M2 smoke test (5 min): check this output first

In [7]:
!python -m floodsitrep.cv.train --config configs/cv/m2_siamese_segformer.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set run_name=smoke_m2 loss=ce lr=6e-4 encoder_lr_mult=0.1 epochs=1 max_train_tiles=320 max_val_tiles=96 log_every=10

run smoke_m2: train 320 tiles, val 96 tiles, device cuda, amp True
config.json: 100% 70.0k/70.0k [00:00<00:00, 79.3MB/s]

pytorch_model.bin: downloading bytes:  81% 44.5M/54.7M [00:01<00:00, 44.2MB/s, 2.76MB/s  ]
pytorch_model.bin: downloading bytes: 100% 51.4M/51.4M [00:02<00:00, 25.0MB/s, 4.84MB/s  ]
pytorch_model.bin: reconstructing file: 100% 54.7M/54.7M [00:02<00:00, 26.6MB/s, 5.21MB/s  ]
Loading weights: 100% 192/192 [00:00<00:00, 17581.57it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b1
Key               | Status     |  | 
------------------+------------+--+-
classifier.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

model.safetensors: downloading bytes:  94% 51.3M/54.7M [00:02<00:00, 44.7MB/s, 3.61MB/s  ]
model.safetensors: downloading bytes: 100% 51.3M/51.3M [00:02<00:00, 19.6MB/s, 4.82MB/s  ]
model.safetensors: reconstructi

## M0 early fusion, seeds 1 and 2 (same settings as seed 0; only the seed changes)

In [8]:
!python -m floodsitrep.cv.train --config configs/cv/m0_early_fusion_unet.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set seed=1

run m0_early_fusion_unet_s1: train 5488 tiles, val 2184 tiles, device cuda, amp True
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 240MB/s]
  epoch 0 step 50/686 loss 0.9718 (running 1.0101)
  epoch 0 step 100/686 loss 0.8706 (running 0.9723)
  epoch 0 step 150/686 loss 0.8468 (running 0.9350)
  epoch 0 step 200/686 loss 0.7680 (running 0.9059)
  epoch 0 step 250/686 loss 0.7520 (running 0.8854)
  epoch 0 step 300/686 loss 0.7666 (running 0.8692)
  epoch 0 step 350/686 loss 0.7715 (running 0.8562)
  epoch 0 step 400/686 loss 0.7309 (running 0.8474)
  epoch 0 step 450/686 loss 0.8656 (running 0.8406)
  epoch 0 step 500/686 loss 0.7688 (running 0.8335)
  epoch 0 step 550/686 loss 0.6650 (running 0.8265)
  epoch 0 step 600/686 loss 0.8610 (running 0.8204)
  epoch 0 step 650/686 loss 0.8737 (running 0.8145)
epoch 0: train_loss 0.8114 val_loss 0.7363 xview2_pixel 0.2071 loc_

In [9]:
!python -m floodsitrep.cv.train --config configs/cv/m0_early_fusion_unet.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set seed=2 num_workers=8

run m0_early_fusion_unet_s2: train 5488 tiles, val 2184 tiles, device cuda, amp True
  epoch 0 step 50/686 loss 0.9939 (running 1.0243)
  epoch 0 step 100/686 loss 0.9283 (running 0.9876)
  epoch 0 step 150/686 loss 0.8904 (running 0.9530)
  epoch 0 step 200/686 loss 0.8039 (running 0.9258)
  epoch 0 step 250/686 loss 0.7763 (running 0.9038)
  epoch 0 step 300/686 loss 0.8625 (running 0.8853)
  epoch 0 step 350/686 loss 0.8556 (running 0.8729)
  epoch 0 step 400/686 loss 0.7679 (running 0.8617)
  epoch 0 step 450/686 loss 0.7403 (running 0.8523)
  epoch 0 step 500/686 loss 0.7346 (running 0.8448)
  epoch 0 step 550/686 loss 0.7236 (running 0.8369)
  epoch 0 step 600/686 loss 0.7936 (running 0.8308)
  epoch 0 step 650/686 loss 0.7480 (running 0.8256)
epoch 0: train_loss 0.8223 val_loss 0.7563 xview2_pixel 0.1888 loc_f1 0.6295 dmg_f1 minor 0.034 major 0.017 destroyed 0.000 (156.5s)
  epoch 1 step 50/686 loss 0.6840 (running 0.7654)
  epoch 1 step 100/686 loss 0.7479 (running 0.7557)
  ep

## M2 siamese SegFormer, weighted CE, seed 0 (optional)
Weighted CE so it compares with the main model (M1, weighted CE). Encoder lr 6e-5, decoder lr 6e-4 (usual SegFormer fine-tuning).

In [10]:
!python -m floodsitrep.cv.train --config configs/cv/m2_siamese_segformer.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}" --set run_name=m2_siamese_segformer_ce_s0 loss=ce lr=6e-4 encoder_lr_mult=0.1 num_workers=8

run m2_siamese_segformer_ce_s0: train 5488 tiles, val 2184 tiles, device cuda, amp True
Loading weights: 100% 192/192 [00:00<00:00, 16811.89it/s]
[transformers] SegformerModel LOAD REPORT from: nvidia/mit-b1
Key               | Status     |  | 
------------------+------------+--+-
classifier.bias   | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  epoch 0 step 50/686 loss 1.2579 (running 1.5277)
  epoch 0 step 100/686 loss 0.9072 (running 1.3576)
  epoch 0 step 150/686 loss 0.4044 (running 1.1690)
  epoch 0 step 200/686 loss 0.7600 (running 1.0230)
  epoch 0 step 250/686 loss 0.5662 (running 0.9300)
  epoch 0 step 300/686 loss 0.9525 (running 0.8558)
  epoch 0 step 350/686 loss 0.2226 (running 0.7941)
  epoch 0 step 400/686 loss 0.3476 (running 0.7454)
  epoch 0 step 450/686 loss 0.2586 (running 0.7028)
  epoch 0 step 500/686 loss 0.2209 (running 0.6702)
  ep

In [11]:
import json, os, pandas as pd
CK = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT/checkpoints"
FAMILIES = {
    "M1 weighted CE":    "m1_siamese_unet_ce",
    "M1 Dice+focal":     "m1_siamese_unet",
    "M0 Dice+focal":     "m0_early_fusion_unet",
    "M2 SegFormer CE":   "m2_siamese_segformer_ce",
    "M2 SegFormer ADE CE": "m2_siamese_segformer_ade_ce",
}
rows = []
for name, pre in FAMILIES.items():
    for s in range(3):
        p = f"{CK}/{pre}_s{s}/metrics.jsonl"
        if not os.path.exists(p):
            continue
        df = pd.DataFrame([json.loads(l) for l in open(p)])
        b = df.loc[df["xview2_pixel_score"].idxmax()]
        rows.append({"model": name, "seed": s, "epochs_run": len(df), "best_epoch": int(b.epoch),
                     "score": b.xview2_pixel_score, "loc": b.loc_f1, "minor": b.dmg_f1_minor,
                     "major": b.dmg_f1_major, "destroyed": b.dmg_f1_destroyed,
                     "s_per_epoch": round(df.seconds.median())})
T = pd.DataFrame(rows)
print(T.round(3).to_string(index=False))
print()
print(T.groupby("model", sort=False)[["score", "loc", "minor", "major", "destroyed"]].agg(["mean", "std", "count"]).round(3))

          model  seed  epochs_run  best_epoch  score   loc  minor  major  destroyed  s_per_epoch
 M1 weighted CE     0          20          12  0.363 0.782  0.100  0.449      0.124          181
 M1 weighted CE     1          20          12  0.363 0.803  0.086  0.519      0.123          369
 M1 weighted CE     2          20          18  0.356 0.803  0.100  0.532      0.090          368
  M1 Dice+focal     0          20           9  0.316 0.794  0.112  0.425      0.043          193
  M1 Dice+focal     1          20          14  0.326 0.819  0.107  0.524      0.045          368
  M1 Dice+focal     2          20          11  0.313 0.787  0.099  0.598      0.044          370
  M0 Dice+focal     0          20           9  0.286 0.756  0.080  0.507      0.032          144
  M0 Dice+focal     1          20           7  0.274 0.760  0.081  0.544      0.022          379
  M0 Dice+focal     2          20           8  0.268 0.751  0.093  0.509      0.019          144
M2 SegFormer CE     0         

In [12]:
df = pd.DataFrame([json.loads(l) for l in open(f"{CK}/m2_siamese_segformer_ce_s0/metrics.jsonl")])
print(df[["epoch", "train_loss", "val_loss", "xview2_pixel_score", "loc_f1",
          "dmg_f1_minor", "dmg_f1_major", "dmg_f1_destroyed"]].round(3).to_string(index=False))

 epoch  train_loss  val_loss  xview2_pixel_score  loc_f1  dmg_f1_minor  dmg_f1_major  dmg_f1_destroyed
     0       0.591     0.147               0.253   0.747         0.016         0.584             0.031
     1       0.312     0.130               0.308   0.720         0.083         0.492             0.066
     2       0.283     0.138               0.362   0.774         0.166         0.599             0.080
     3       0.259     0.130               0.341   0.803         0.146         0.642             0.054
     4       0.253     0.142               0.268   0.802         0.131         0.738             0.011
     5       0.230     0.146               0.319   0.802         0.063         0.780             0.058
     6       0.220     0.140               0.326   0.814         0.133         0.705             0.042
     7       0.212     0.144               0.340   0.805         0.128         0.747             0.055
     8       0.195     0.144               0.296   0.796         0.098   

## Stop billing

In [13]:
from google.colab import runtime
runtime.unassign()